# Laser Position PV Archiver Probe

For each `laser / Position` PV marked `measurement: true` in `pv_groups.yaml`, probe both the base name and the `1H`-sampled name against the archive appliance. All helpers now live in `sparklines_v2.archive.probe` and `sparklines_v2.archive.hierarchy`.

In [ ]:
import datetime as dt

try:
    import pandas as pd
except ImportError:
    pd = None

from sparklines_v2.archive.probe import (
    base_and_1h_candidates,
    iter_laser_position_measurements,
    probe_archive_pv,
)
from sparklines_v2.beamlines import LOCAL_TIMEZONE

configured_pvs = list(iter_laser_position_measurements())
probe_specs = [
    candidate
    for pv in configured_pvs
    for candidate in base_and_1h_candidates(pv)
]
configured_pvs, probe_specs

In [ ]:
end = dt.datetime.now(tz=LOCAL_TIMEZONE)
start = end - dt.timedelta(hours=1)

rows = []
for spec in probe_specs:
    row = dict(spec)
    try:
        row.update(probe_archive_pv(spec["pv_name"], start, end, timeout=20.0))
    except Exception as exc:
        row.update({"ok": False, "point_count": 0, "elapsed_s": None, "error": repr(exc)})
    rows.append(row)

if pd is not None:
    results = pd.DataFrame(rows)
    display(results)
else:
    results = rows
    for row in rows:
        print(row)

In [ ]:
if pd is not None:
    display(
        results.pivot_table(
            index="configured_pv",
            columns="candidate_kind",
            values="point_count",
            aggfunc="first",
            fill_value=0,
        )
    )